In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "credit_card_project.silver.cleaned_transactions"
GOLD_TABLE = "credit_card_project.gold.gold_merchant_behavior"

silver_df = spark.table(SILVER_TABLE)

print("Silver schema:")
silver_df.printSchema()

print(f"Silver row count: {silver_df.count():,}")


### Inspect merchant-related fields
Which fields we need from Silver? Confirm the merchant population before writing the aggregation

In [0]:
merchant_related_cols = [
    c for c in silver_df.columns
    if any(term in c.lower() for term in [
        "merchant",
        "mcc",
        "city",
        "state",
        "amount",
        "fraud",
        "online",
        "timestamp",
        "user",
        "card"
    ])
]

print("Relevant columns:")
for c in merchant_related_cols:
    print(f" - {c}")

In [0]:
silver_df.select(*merchant_related_cols).limit(10).show(
    truncate=False,
    vertical=True
)

### Confirm merchant population

In [0]:
from pyspark.sql import functions as F

# Merchant population
merchant_population = (
    silver_df
    .select("merchant_id")
    .distinct()
)

print(f"Distinct merchants: {merchant_population.count():,}")

# NULL merchant IDs
silver_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("merchant_id").isNull().cast("long")).alias("null_merchant_ids")
).show()

# Merchant → MCC cardinality
merchant_mcc_counts = (
    silver_df
    .where(F.col("merchant_id").isNotNull())
    .groupBy("merchant_id")
    .agg(F.countDistinct("merchant_category_code").alias("mcc_count"))
)
# Take the per-merchant counts and create a frequency distribution:
merchant_mcc_counts.groupBy("mcc_count").count().orderBy("mcc_count").show() 

# Location NULL characteristics
silver_df.select(
    F.sum(F.col("merchant_city").isNull().cast("long")).alias("null_city"),
    F.sum(F.col("merchant_state").isNull().cast("long")).alias("null_state")
).show()


### Build the merchant aggregation

In [0]:
merchant_agg = (
    silver_df
    .where(F.col("merchant_id").isNotNull())
    .groupBy("merchant_id")
    .agg(
        F.count("*").alias("transaction_count"),

        F.sum("amount").alias("total_amount"),

        F.avg("amount").alias("avg_transaction_amount"),

        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        ).alias("fraud_count"),

        F.countDistinct("user_id").alias("unique_users"),

        F.countDistinct("card_id").alias("unique_cards"),

        F.countDistinct("merchant_category_code").alias("unique_mccs"),

        F.countDistinct("merchant_city").alias("unique_cities"),

        F.countDistinct("merchant_state").alias("unique_states"),

        (
            F.avg(
                F.when(F.col("is_online") == True, 1.0)
                 .otherwise(0.0)
            ) * 100
        ).alias("online_transaction_pct")
    )
)


In [0]:
merchant_agg = (
    merchant_agg
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("fraud_count") / F.col("transaction_count")
        )
    )
    .select(
        "merchant_id",
        "transaction_count",
        "total_amount",
        "avg_transaction_amount",
        "fraud_count",
        "fraud_rate",
        "unique_users",
        "unique_cards",
        "unique_mccs",
        "unique_cities",
        "unique_states",
        "online_transaction_pct"
    )
)


###  Check the aggregation before writing

In [0]:
print(f"Merchant aggregation rows: {merchant_agg.count():,}")

merchant_agg.printSchema()

merchant_agg.show(10, truncate=False)


Validate the key reconciliations:

In [0]:
merchant_agg.select(
    F.sum("transaction_count").alias("transaction_count"),
    F.sum("total_amount").alias("total_amount"),
    F.sum("fraud_count").alias("fraud_count")
).show()


Expected:
- transaction_count = 24,386,900
- total_amount      = 1,064,098,117.16
- fraud_count       = 29,757


In [0]:
merchant_agg.select(
    F.count("*").alias("merchant_rows"),
    F.countDistinct("merchant_id").alias("distinct_merchants"),
    F.sum(F.when(F.col("merchant_id").isNull(), 1).otherwise(0)).alias("null_keys")
).show()


Expected:
- merchant_rows       = 100,343
- distinct_merchants  = 100,343
- null_keys           = 0


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

SILVER_TABLE = "credit_card_project.silver.cleaned_transactions"
GOLD_TABLE = "credit_card_project.gold.gold_merchant_behavior"

EXPECTED_MERCHANTS = 100_343
EXPECTED_TRANSACTIONS = 24_386_900
EXPECTED_FRAUD_COUNT = 29_757
EXPECTED_TOTAL_AMOUNT = 1_064_098_117.16


In [0]:
silver_df = spark.table(SILVER_TABLE)

print(f"Silver rows: {silver_df.count():,}")


###  Build merchant-level aggregation

In [0]:
merchant_agg = (
    silver_df
    .where(F.col("merchant_id").isNotNull())
    .groupBy("merchant_id")
    .agg(
        F.count("*").cast("long").alias("transaction_count"),

        F.sum("amount").cast("decimal(18,2)").alias("total_amount"),

        F.avg("amount").cast("decimal(18,4)").alias("avg_transaction_amount"),

        F.sum(
            F.when(F.col("is_fraud") == True, 1).otherwise(0)
        ).cast("long").alias("fraud_count"),

        F.countDistinct("user_id").cast("long").alias("unique_users"),

        F.countDistinct("card_id").cast("long").alias("unique_cards"),

        F.countDistinct("merchant_category_code").cast("long").alias("unique_mccs"),

        F.countDistinct("merchant_city").cast("long").alias("unique_cities"),

        F.countDistinct("merchant_state").cast("long").alias("unique_states"),

        (
            F.avg(
                F.when(F.col("is_online") == True, 1.0)
                 .otherwise(0.0)
            ) * 100.0
        ).cast("double").alias("online_transaction_pct")
    )
    .withColumn(
        "fraud_rate",
        F.when(
            F.col("transaction_count") > 0,
            (
                F.col("fraud_count") /
                F.col("transaction_count")
            ).cast("double")
        )
    )
    .select(
        "merchant_id",
        "transaction_count",
        "total_amount",
        "avg_transaction_amount",
        "fraud_count",
        "fraud_rate",
        "unique_users",
        "unique_cards",
        "unique_mccs",
        "unique_cities",
        "unique_states",
        "online_transaction_pct"
    )
)


### Validate the in-memory result

In [0]:
merchant_agg.select(
    F.count("*").alias("merchant_rows"),
    F.countDistinct("merchant_id").alias("distinct_merchants"),
    F.sum("transaction_count").alias("transactions"),
    F.sum("total_amount").alias("total_amount"),
    F.sum("fraud_count").alias("fraud_count")
).show()


Expected:
- merchant_rows       100343
- distinct_merchants  100343
- transactions        24386900
- total_amount        1064098117.16
- fraud_count         29757


In [0]:
merchant_agg.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate"),
    F.min("online_transaction_pct").alias("min_online_pct"),
    F.max("online_transaction_pct").alias("max_online_pct")
).show()


expected:
- fraud_rate              >= 0 and <= 1
- online_transaction_pct  >= 0 and <= 100


### Persist the Gold table

In [0]:
(
    merchant_agg
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)


In [0]:
gold_df = spark.table(GOLD_TABLE)

gold_df.printSchema()
print(f"Persisted Gold rows: {gold_df.count():,}")


### Final persisted-table validation
Validate the actual Delta table, not just the DataFrame that produced it

In [0]:
from pyspark.sql import functions as F

# Core reconciliation
validation = gold_df.select(
    F.count("*").alias("merchant_rows"),
    F.countDistinct("merchant_id").alias("distinct_merchants"),
    F.sum("transaction_count").alias("transactions"),
    F.sum("total_amount").alias("total_amount"),
    F.sum("fraud_count").alias("fraud_count")
).collect()[0]

# Duplicate keys
duplicate_merchants = (
    gold_df
    .groupBy("merchant_id")
    .count()
    .where(F.col("count") > 1)
    .count()
)

# Percentage ranges
ranges = gold_df.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate"),
    F.min("online_transaction_pct").alias("min_online_pct"),
    F.max("online_transaction_pct").alias("max_online_pct")
).collect()[0]

checks = {
    "merchant_population":
        validation["merchant_rows"] == EXPECTED_MERCHANTS,

    "distinct_merchant_keys":
        validation["distinct_merchants"] == EXPECTED_MERCHANTS,

    "duplicate_merchant_keys":
        duplicate_merchants == 0,

    "transaction_reconciliation":
        validation["transactions"] == EXPECTED_TRANSACTIONS,

    "spend_reconciliation":
        float(validation["total_amount"]) == EXPECTED_TOTAL_AMOUNT,

    "fraud_reconciliation":
        validation["fraud_count"] == EXPECTED_FRAUD_COUNT,

    "fraud_rate_range":
        ranges["min_fraud_rate"] >= 0
        and ranges["max_fraud_rate"] <= 1,

    "online_pct_range":
        ranges["min_online_pct"] >= 0
        and ranges["max_online_pct"] <= 100,
}

print("=== GOLD MERCHANT BEHAVIOR VALIDATION ===")
print(f"Merchant rows:       {validation['merchant_rows']:,}")
print(f"Distinct merchants:  {validation['distinct_merchants']:,}")
print(f"Duplicate keys:      {duplicate_merchants}")
print(f"Transactions:        {validation['transactions']:,}")
print(f"Total amount:        {validation['total_amount']}")
print(f"Fraud count:         {validation['fraud_count']:,}")
print()
print(f"Fraud rate range:    {ranges['min_fraud_rate']} → {ranges['max_fraud_rate']}")
print(f"Online pct range:    {ranges['min_online_pct']} → {ranges['max_online_pct']}")
print()

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

all_passed = all(checks.values())

print()
print(f"FINAL PERSISTED RESULT: {'PASS' if all_passed else 'FAIL'}")

assert all_passed, "Persisted Gold merchant behavior validation failed."



In [0]:
gold_df.select(
    F.min("unique_cities").alias("min_unique_cities"),
    F.max("unique_cities").alias("max_unique_cities"),
    F.min("unique_states").alias("min_unique_states"),
    F.max("unique_states").alias("max_unique_states"),
    F.min("unique_mccs").alias("min_unique_mccs"),
    F.max("unique_mccs").alias("max_unique_mccs")
).show()


### Engineering review

In [0]:
gold_df.printSchema()


### Basic data sanity
expected: all 0

In [0]:
gold_df.select(
    F.sum(F.when(F.col("merchant_id").isNull(), 1).otherwise(0))
        .alias("null_merchant_ids"),

    F.sum(F.when(F.col("transaction_count") <= 0, 1).otherwise(0))
        .alias("invalid_transaction_counts"),

    F.sum(F.when(F.col("fraud_count") < 0, 1).otherwise(0))
        .alias("invalid_fraud_counts"),

    F.sum(
        F.when(
            (F.col("fraud_count") > F.col("transaction_count")) |
            (F.col("fraud_rate") < 0) |
            (F.col("fraud_rate") > 1),
            1
        ).otherwise(0)
    ).alias("invalid_fraud_metrics"),

    F.sum(
        F.when(
            (F.col("online_transaction_pct") < 0) |
            (F.col("online_transaction_pct") > 100),
            1
        ).otherwise(0)
    ).alias("invalid_online_pct")
).show()


In [0]:
gold_df.groupBy("unique_mccs").count().orderBy("unique_mccs").show()


Multiple locations are valid because location belongs to the transaction observation, not permanently to merchant_id:

In [0]:
gold_df.groupBy("unique_cities").count().orderBy("unique_cities").show()

gold_df.groupBy("unique_states").count().orderBy("unique_states").show()
